# Лабораторная работа № 1

Анализ публикаций arXiv о Retrieval-Augmented Generation (RAG): исследовательский анализ, ключевые слова, графы и поиск.

В ноутбук включён код всех шести Python-файлов, включая `common.py`. Выполняйте ячейки сверху вниз. Исходные `.py` для запуска не требуются.

По умолчанию используется существующий `data/papers.csv`. Для повторного сбора из arXiv установите `RECOLLECT = True` (нужен интернет). Результаты сохраняются в `data/`, `figures/` и `results/`; графики также выводятся в ноутбуке.

## 0. Подготовка

Откройте ноутбук из папки `лаб1` или из корня проекта. Выберите Python-окружение с перечисленными ниже библиотеками. При необходимости выполните установку, убрав `#` перед `%pip`.

In [ ]:
# %pip install pandas numpy matplotlib seaborn requests nltk scikit-learn networkx scipy


### Общие функции

Загрузка корпуса, токенизация, нормализация и извлечение n-грамм (`common.py`).

In [ ]:
"""Общие утилиты: загрузка корпуса, токенизация, нормализация, извлечение n-грамм."""
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

ROOT = Path.cwd()
if ROOT.name != "лаб1" and (ROOT / "лаб1").is_dir():
    ROOT = ROOT / "лаб1"
DATA = ROOT / "data"
FIG = ROOT / "figures"
RES = ROOT / "results"
for d in (DATA, FIG, RES):
    d.mkdir(exist_ok=True)

BOILERPLATE = set("""
paper papers study studies work works approach approaches method methods propose proposed
present presents introduce introduces show shows shown demonstrate demonstrates result results
using use used uses based novel new existing recent recently however also can may across
within without among via thus including include includes order significantly significant
experiments experimental experiment evaluate evaluation evaluated evaluating performance
improve improves improved improvement improving effective effectiveness different various
address addresses challenge challenges problem problems task tasks case cases large
real world data dataset datasets benchmark benchmarks achieve achieves achieved state art
framework system systems provide provides provided enable enables enabled well high
number time need needs able make makes like often one two three first second
end instance instances objective consistent improvements baseline baselines systematically
investigate investigates fundamental impact enhancing enhance enhances relevant conventional
looking employing employ employs samples sample stage wise key main major simple strong
extensive comprehensive substantial notable remarkable consistently outperform outperforms
outperforming outperformed superior compared compare compares demonstrating demonstrated
leverage leverages leveraging incorporate incorporates incorporating integrate integrates
integrating particularly specifically especially despite although while whether suffer suffers
limited limitation limitations lack lacks leading leads lead effectively efficiently
advance advances advanced develop develops developed design designed designs apply applying
applied application applications aim aims aimed focus focuses approach potential
findings underscore underscores wide range percentage points remains underexplored combines
highlight highlights highlighting reveal reveals revealed combine combining offers offer
significantly gains gain trade offs additional additionally further furthermore moreover
widely commonly increasingly rapidly demonstrates establishes establish
""".split())


In [ ]:
STOP = set(ENGLISH_STOP_WORDS) | BOILERPLATE
QUERY_WORDS = {"retrieval", "augmented", "generation", "rag"}

_stem = SnowballStemmer("english").stem
_TOKEN = re.compile(r"[a-z][a-z0-9]+")
_PHRASE = re.compile(r"retrieval[\s-]+augmented(?:[\s-]+generation)?|\brags?\b|\bretrieval[\s-]+enhanced[\s-]+generation")
_MULTI = re.compile(r"\bmulti[\s-]+(?=[a-z])")
_CHUNK = re.compile(r"[.,;:!?()\[\]{}\"“”/|]+")


def load_papers() -> pd.DataFrame:
    df = pd.read_csv(DATA / "papers.csv")
    df["text"] = df["title"] + ". " + df["abstract"]
    df["year"] = df["published"].str[:4].astype(int)
    return df


def tokens(text: str) -> list[str]:
    """Нижний регистр, дефисы -> пробелы, только слова из латиницы."""
    return _TOKEN.findall(text.lower().replace("-", " "))


def stem(w: str) -> str:
    return _stem(w)


def content_runs(text: str) -> list[list[tuple[str, str]]]:
    """Список непрерывных фрагментов из содержательных слов: [(stem, surface), ...].
    Знаки препинания и стоп-слова разрывают фрагмент, поэтому n-граммы
    не пересекают границы фраз."""
    runs = []
    text = _PHRASE.sub(" , ", text.lower())
    text = _MULTI.sub("multi", text).replace("-", " ")
    for chunk in _CHUNK.split(text):
        run = []
        for w in _TOKEN.findall(chunk):
            if w in STOP or len(w) < 3 or w.isdigit():
                if run:
                    runs.append(run)
                run = []
            else:
                run.append((stem(w), w))
        if run:
            runs.append(run)
    return runs


In [ ]:
def ngrams_of(text: str, nmax: int = 3):
    """Возвращает Counter {tuple(stems): count} и Counter поверхностных форм."""
    counts, surf = Counter(), {}
    for run in content_runs(text):
        for n in range(1, nmax + 1):
            for i in range(len(run) - n + 1):
                gram = run[i:i + n]
                key = tuple(s for s, _ in gram)
                if n == 1 and key[0] in {stem(w) for w in QUERY_WORDS}:
                    continue
                counts[key] += 1
                surf.setdefault(key, Counter())[" ".join(w for _, w in gram)] += 1
    return counts, surf


## 1. Сбор публикаций

Загрузка метаданных из arXiv или использование сохранённого корпуса.

Исходный файл: `01_collect.py`.

In [ ]:
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import requests

API = "https://export.arxiv.org/api/query"
NS = {"a": "http://www.w3.org/2005/Atom"}
QUERY = ('(ti:"retrieval-augmented generation" OR abs:"retrieval-augmented generation" '
         'OR abs:"retrieval augmented generation") '
         'AND submittedDate:[202001010000 TO 202612312359]')
TARGET = 800
PAGE = 100
OUT = DATA / "papers.csv"


def fetch(start: int) -> list[dict]:
    params = {"search_query": QUERY, "start": start, "max_results": PAGE,
              "sortBy": "relevance", "sortOrder": "descending"}
    for attempt in range(5):
        r = requests.get(API, params=params, timeout=60)
        if r.status_code == 200:
            break
        time.sleep(5 * (attempt + 1))
    r.raise_for_status()
    root = ET.fromstring(r.content)
    rows = []
    for e in root.findall("a:entry", NS):
        rows.append({
            "id": e.find("a:id", NS).text.strip(),
            "title": " ".join(e.find("a:title", NS).text.split()),
            "authors": "; ".join(a.find("a:name", NS).text for a in e.findall("a:author", NS)),
            "abstract": " ".join(e.find("a:summary", NS).text.split()),
            "published": e.find("a:published", NS).text[:10],
            "category": e.find("{http://arxiv.org/schemas/atom}primary_category").get("term"),
        })
    return rows


In [ ]:
def main():
    OUT.parent.mkdir(exist_ok=True)
    rows = []
    for start in range(0, TARGET, PAGE):
        batch = fetch(start)
        print(f"start={start}: получено {len(batch)}")
        rows += batch
        if len(batch) < PAGE:
            break
        time.sleep(3)
    df = pd.DataFrame(rows).drop_duplicates("id")
    df.to_csv(OUT, index=False)
    print(f"Сохранено {len(df)} публикаций, {df.published.min()} .. {df.published.max()} -> {OUT}")


RECOLLECT = False
if RECOLLECT or not OUT.exists():
    main()
else:
    print(f"Используется сохранённый корпус: {OUT}")


## 2. Исследовательский анализ данных

Динамика публикаций, категории, авторы, длины текстов, законы Ципфа и Хипса.

Исходный файл: `02_eda.py`.

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


sns.set_theme(style="whitegrid", context="talk")
df = load_papers()
out = {}

df["n_authors"] = df.authors.str.count(";") + 1
df["abs_words"] = df.abstract.map(lambda s: len(tokens(s)))
df["title_words"] = df.title.map(lambda s: len(tokens(s)))
out["n_papers"] = len(df)
out["date_min"], out["date_max"] = df.published.min(), df.published.max()
out["abs_words_mean"] = float(df.abs_words.mean())
out["abs_words_median"] = float(df.abs_words.median())
out["title_words_mean"] = float(df.title_words.mean())
out["authors_mean"] = float(df.n_authors.mean())
out["by_year"] = df.year.value_counts().sort_index().to_dict()
out["top_categories"] = df.category.value_counts().head(10).to_dict()

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
df.year.value_counts().sort_index().plot.bar(ax=ax[0], color="#4C72B0")
ax[0].set(title="Публикации по годам", xlabel="Год", ylabel="Число статей")
q = df.published.str[:7].value_counts().sort_index()
q = q[q.index >= "2023-01"]
q.plot(ax=ax[1], marker="o", color="#DD8452")
ax[1].set(title="Публикации по месяцам (с 2023)", xlabel="Месяц", ylabel="Число статей")
ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(FIG / "01_dynamics.png", dpi=150); plt.show(); plt.close()

fig, ax = plt.subplots(1, 3, figsize=(20, 6))


df.category.value_counts().head(10).sort_values().plot.barh(ax=ax[0], color="#55A868")
ax[0].set(title="Топ-10 категорий arXiv", xlabel="Статей")
sns.histplot(df.abs_words, bins=30, ax=ax[1], color="#C44E52")
ax[1].axvline(df.abs_words.median(), color="k", ls="--")
ax[1].set(title="Длина аннотации (слов)", xlabel="Слов", ylabel="Статей")
sns.histplot(df.n_authors.clip(upper=20), bins=20, ax=ax[2], color="#8172B2")
ax[2].set(title="Авторов на статью (обрезано на 20)", xlabel="Авторов", ylabel="Статей")
plt.tight_layout(); plt.savefig(FIG / "02_categories_lengths.png", dpi=150); plt.show(); plt.close()


In [ ]:
authors = Counter(a for s in df.authors for a in s.split("; "))
out["n_unique_authors"] = len(authors)
out["top_authors"] = authors.most_common(15)
top = authors.most_common(15)[::-1]
plt.figure(figsize=(10, 8))
plt.barh([a for a, _ in top], [c for _, c in top], color="#64B5CD")
plt.title("Самые продуктивные авторы"); plt.xlabel("Статей")
plt.tight_layout(); plt.savefig(FIG / "03_authors.png", dpi=150); plt.show(); plt.close()

raw = Counter(w for t in df.text for w in tokens(t))
content = Counter(s for t in df.text for run in content_runs(t) for s, _ in run)
bigrams = Counter()
for t in df.text:
    for run in content_runs(t):
        stems = [s for s, _ in run]
        bigrams.update(zip(stems, stems[1:]))


def zipf_fit(counter, kmin=1, kmax=None):
    freqs = np.array(sorted(counter.values(), reverse=True), dtype=float)
    ranks = np.arange(1, len(freqs) + 1)
    sl = slice(kmin - 1, kmax)
    slope, intercept = np.polyfit(np.log10(ranks[sl]), np.log10(freqs[sl]), 1)
    return ranks, freqs, slope, intercept


fig, ax = plt.subplots(1, 3, figsize=(21, 6.5))


for a, (name, c, kmax) in zip(ax, [("Все слова (сырой текст)", raw, 1000),
                                   ("Содержательные слова (стемы)", content, 1000),
                                   ("Биграммы содержательных слов", bigrams, 1000)]):
    ranks, freqs, slope, ic = zipf_fit(c, 1, kmax)
    a.loglog(ranks, freqs, ".", alpha=.6, label="эмпирика")
    a.loglog(ranks, 10 ** ic * ranks ** slope, "r-", lw=2, label=f"fit: наклон = {slope:.2f}")
    a.loglog(ranks, freqs[0] * ranks ** -1.0, "k--", lw=1, label="идеал Ципфа (наклон −1)")
    a.set(title=name, xlabel="Ранг", ylabel="Частота"); a.legend()
    key = {"Все слова (сырой текст)": "zipf_raw", "Содержательные слова (стемы)": "zipf_content",
           "Биграммы содержательных слов": "zipf_bigrams"}[name]
    out[key] = {"slope": float(slope), "vocab": len(c), "tokens": int(sum(c.values()))}
plt.tight_layout(); plt.savefig(FIG / "04_zipf.png", dpi=150); plt.show(); plt.close()


In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(df))
seen, vocab_curve, tok_curve, n = set(), [], [], 0
for i in order:
    for w in tokens(df.text.iloc[i]):
        n += 1; seen.add(w)
    tok_curve.append(n); vocab_curve.append(len(seen))
k, logb = np.polyfit(np.log10(tok_curve), np.log10(vocab_curve), 1)
out["heaps_beta"] = float(k); out["heaps_K"] = float(10 ** logb)
plt.figure(figsize=(8, 6))
plt.loglog(tok_curve, vocab_curve, label="эмпирика")
plt.loglog(tok_curve, 10 ** logb * np.array(tok_curve) ** k, "r--", label=f"K·N^β, β={k:.2f}")
plt.xlabel("Токенов N"); plt.ylabel("Словарь V"); plt.title("Закон Хипса"); plt.legend()
plt.tight_layout(); plt.savefig(FIG / "05_heaps.png", dpi=150); plt.show(); plt.close()

top_words = Counter({w: c for w, c in raw.items() if w not in STOP and len(w) > 2}).most_common(25)
out["top_words"] = top_words
plt.figure(figsize=(10, 9))
plt.barh([w for w, _ in top_words][::-1], [c for _, c in top_words][::-1], color="#937860")
plt.title("Топ-25 слов (без стоп-слов)"); plt.xlabel("Частота")
plt.tight_layout(); plt.savefig(FIG / "06_top_words.png", dpi=150); plt.show(); plt.close()

(RES / "eda.json").write_text(json.dumps(out, ensure_ascii=False, indent=2, default=int))


print(json.dumps(out, ensure_ascii=False, indent=2, default=int))


## 3. Извлечение ключевых слов

Отбор n-грамм по частоте документов и TF-IDF с учётом заголовков.

Исходный файл: `03_keywords.py`.

In [ ]:
import json
import math
from collections import Counter


MIN_DF = 4          
MAX_DF_RATIO = 0.35  
TOP_K = 12          
TITLE_BOOST = 2     

df = load_papers()
N = len(df)

per_doc, surface = [], {}
for _, r in df.iterrows():
    c, s = ngrams_of(r.abstract)
    t, st = ngrams_of(r.title)
    for k, v in t.items():
        c[k] += v * (TITLE_BOOST - 1)
    for src in (s, st):
        for k, cnt in src.items():
            surface.setdefault(k, Counter()).update(cnt)
    per_doc.append(c)

dfreq = Counter(k for c in per_doc for k in c)
vocab = {k for k, d in dfreq.items() if MIN_DF <= d <= MAX_DF_RATIO * N}
print(f"кандидатов всего: {len(dfreq)}, после фильтра по df: {len(vocab)}")


def label(key):
    return surface[key].most_common(1)[0][0]


def contained(a, b):
    """a — подпоследовательность b (подряд идущие стемы)"""
    return len(a) < len(b) and any(b[i:i + len(a)] == a for i in range(len(b) - len(a) + 1))


In [ ]:
keywords = []
for c in per_doc:
    scored = []
    for k, tf in c.items():
        if k not in vocab:
            continue
        idf = math.log(N / dfreq[k])
        scored.append((tf * idf * (1 + 0.25 * (len(k) - 1)), k))
    scored.sort(reverse=True)
    cand = [k for _, k in scored[:TOP_K * 2]]
    kept = [k for k in cand if not any(contained(k, o) for o in cand)]
    keywords.append([label(k) for k in kept[:TOP_K]])

df["keywords"] = keywords
(DATA / "keywords.json").write_text(json.dumps(
    [{"id": i, "title": t, "keywords": k} for i, t, k in zip(df.id, df.title, keywords)],
    ensure_ascii=False, indent=1))

freq = Counter(k for ks in keywords for k in ks)
n_by_len = Counter(len(k.split()) for k in freq)
stats = {"unique_keywords": len(freq), "by_ngram_len": dict(n_by_len),
         "avg_per_paper": sum(map(len, keywords)) / N, "top30": freq.most_common(30)}
(RES / "keywords_stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2))
print(json.dumps(stats, ensure_ascii=False, indent=1))
for i in range(5):
    print("\n", df.title.iloc[i], "\n  ->", keywords[i])


## 4. Граф ключевых слов

Совместная встречаемость, выделение сообществ, центральности и визуализация.

Исходный файл: `04_keyword_graph.py`.

In [ ]:
import itertools
import json
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from networkx.algorithms import community as nxc


sns.set_theme(style="white", context="talk")
MIN_FREQ = 3
MIN_EDGE_W = 2 
papers = json.loads((DATA / "keywords.json").read_text())

freq = Counter(k for p in papers for k in set(p["keywords"]))
nodes = sorted(k for k, c in freq.items() if c >= MIN_FREQ)
G = nx.Graph()
for k in nodes:
    G.add_node(k, freq=freq[k])
for p in papers:
    ks = sorted(set(p["keywords"]) & set(nodes))
    for a, b in itertools.combinations(ks, 2):
        w = G.get_edge_data(a, b, {"weight": 0})["weight"]
        G.add_edge(a, b, weight=w + 1)
full_Q = nxc.modularity(G, nxc.louvain_communities(G, weight="weight", seed=42), weight="weight")
full_info = {"nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "louvain_modularity": full_Q}
G.remove_edges_from([(u, v) for u, v, d in G.edges(data=True) if d["weight"] < MIN_EDGE_W])
G.remove_nodes_from(list(nx.isolates(G)))
comps = sorted(nx.connected_components(G), key=len, reverse=True)
H = G.subgraph(comps[0]).copy()  
info = {"full_graph": full_info, "nodes": G.number_of_nodes(), "edges": G.number_of_edges(),
        "components": len(comps), "largest_component_nodes": H.number_of_nodes(),
        "largest_component_edges": H.number_of_edges(),
        "density": nx.density(H), "avg_clustering": nx.average_clustering(H),
        "avg_degree": 2 * H.number_of_edges() / H.number_of_nodes()}


In [ ]:
print(info)

algos = {
    "Louvain": lambda g: nxc.louvain_communities(g, weight="weight", resolution=1.0, seed=42),
    "Greedy modularity": lambda g: nxc.greedy_modularity_communities(g, weight="weight"),
    "Label propagation": lambda g: list(nxc.asyn_lpa_communities(g, weight="weight", seed=42)),
}
quality = {}
partitions = {}
for name, f in algos.items():
    parts = [set(c) for c in f(H)]
    partitions[name] = parts
    quality[name] = {"modularity": nxc.modularity(H, parts, weight="weight"),
                     "n_clusters": len(parts),
                     "coverage": nxc.partition_quality(H, parts)[0],
                     "largest_cluster": max(len(c) for c in parts),
                     "n_clusters_ge5": sum(len(c) >= 5 for c in parts)}

null_q = []
deg = [d for _, d in H.degree()]
for s in range(5):
    R = nx.Graph(nx.configuration_model(deg, seed=s))
    R.remove_edges_from(nx.selfloop_edges(R))
    null_q.append(nxc.modularity(R, nxc.louvain_communities(R, seed=s)))
quality["null_model_modularity_mean"] = float(np.mean(null_q))
print(json.dumps(quality, indent=1))

best = partitions["Louvain"]
best = sorted(best, key=len, reverse=True)
node2c = {n: i for i, c in enumerate(best) for n in c}
nx.set_node_attributes(H, node2c, "cluster")


Hw = H.copy()


In [ ]:
for u, v, d in Hw.edges(data=True):
    d["dist"] = 1.0 / d["weight"]


cent = pd.DataFrame({
    "freq": pd.Series(nx.get_node_attributes(H, "freq")),
    "degree": pd.Series(nx.degree_centrality(H)),
    "weighted_degree": pd.Series(dict(H.degree(weight="weight"))),
    "betweenness": pd.Series(nx.betweenness_centrality(Hw, weight="dist")),
    "closeness": pd.Series(nx.closeness_centrality(Hw, distance="dist")),
    "eigenvector": pd.Series(nx.eigenvector_centrality_numpy(H, weight="weight")),
    "pagerank": pd.Series(nx.pagerank(H, weight="weight")),
    "cluster": pd.Series(node2c),
})
cent.index.name = "keyword"
cent.to_csv(RES / "keyword_centralities.csv")
print(cent.sort_values("degree", ascending=False).head(15).round(3))


cluster_rows = []
for i, c in enumerate(best):
    sub = cent.loc[list(c)]
    top = sub.sort_values("weighted_degree", ascending=False).head(10).index.tolist()
    cluster_rows.append({"cluster": i, "size": len(c), "top_keywords": top,
                         "internal_weight": int(sum(d["weight"] for u, v, d in H.subgraph(c).edges(data=True)))})
(RES / "clusters.json").write_text(json.dumps(cluster_rows, ensure_ascii=False, indent=1))
(RES / "graph_stats.json").write_text(json.dumps({"graph": info, "quality": quality}, indent=1))
for r in cluster_rows[:12]:
    print(r["cluster"], r["size"], r["top_keywords"])



fig, ax = plt.subplots(figsize=(18, 14))
pos = nx.spring_layout(H, k=0.35, seed=7, weight="weight", iterations=150)
palette = sns.color_palette("tab20", n_colors=max(len(best), 3))
sizes = [40 + 12 * cent.loc[n, "weighted_degree"] for n in H]
nx.draw_networkx_edges(H, pos, alpha=0.15, width=[0.4 + 0.5 * d["weight"] for *_, d in H.edges(data=True)], ax=ax)
nx.draw_networkx_nodes(H, pos, node_size=sizes, node_color=[palette[node2c[n] % 20] for n in H],
                       alpha=0.9, linewidths=0.3, edgecolors="k", ax=ax)



labels = {}
for i, c in enumerate(best[:16]):
    labels[cent.loc[list(c)].sort_values("weighted_degree", ascending=False).index[0]] = f"{i}: "
labels = {n: l + n for n, l in labels.items()}
nx.draw_networkx_labels(H, pos, labels, font_size=12, font_weight="bold", ax=ax,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
ax.set_title(f"Граф ключевых слов: {H.number_of_nodes()} вершин, {H.number_of_edges()} рёбер, "
             f"Louvain: {len(best)} кластеров, Q = {quality['Louvain']['modularity']:.3f}")
ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "07_keyword_graph.png", dpi=150); plt.show(); plt.close()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for a, col in zip(axes.ravel(), ["degree", "betweenness", "closeness", "eigenvector"]):
    s = cent[col].sort_values().tail(12)
    a.barh(s.index, s.values, color=sns.color_palette("viridis", 12))
    a.set_title(f"Топ-12: {col}")
plt.tight_layout(); plt.savefig(FIG / "08_centralities.png", dpi=150); plt.show(); plt.close()


fig, ax = plt.subplots(1, 2, figsize=(16, 6))
dseq = np.array(sorted([d for _, d in H.degree()], reverse=True))
cnt = Counter(dseq)
ax[0].loglog(list(cnt.keys()), list(cnt.values()), "o", alpha=.7)
ax[0].set(title="Распределение степеней вершин", xlabel="Степень", ylabel="Число вершин")
ax[1].bar(range(len(best)), [len(c) for c in best], color="#4C72B0")
ax[1].set(title="Размеры кластеров (Louvain)", xlabel="Кластер", ylabel="Ключевых слов")
plt.tight_layout(); plt.savefig(FIG / "09_degree_clusters.png", dpi=150); plt.show(); plt.close()


plt.figure(figsize=(8, 6.5))
sns.heatmap(cent[["degree", "betweenness", "closeness", "eigenvector", "pagerank", "freq"]]
            .corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
plt.title("Корреляция центральностей (Спирмен)")


plt.tight_layout(); plt.savefig(FIG / "10_centrality_corr.png", dpi=150); plt.show(); plt.close()

nx.write_gexf(H, RES / "keyword_graph.gexf")


## 5. Граф публикаций и поиск

Поиск похожих статей, поиск по запросу, персонализированный PageRank и сравнение с TF-IDF.

Исходный файл: `05_paper_graph_search.py`.

In [ ]:
import itertools
import json
import math
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


sns.set_theme(style="white", context="talk")

papers = load_papers()
kw = {p["id"]: p["keywords"] for p in json.loads((DATA / "keywords.json").read_text())}
ids = papers.id.tolist()
title = dict(zip(papers.id, papers.title))
category = dict(zip(papers.id, papers.category))
N = len(ids)

inv = defaultdict(list)                      
for pid, ks in kw.items():
    for k in set(ks):
        inv[k].append(pid)
idf = {k: math.log(N / len(v)) for k, v in inv.items()}

shared = defaultdict(set)                     
for k, ps in inv.items():
    if len(ps) > 60:                           
        continue
    for a, b in itertools.combinations(sorted(ps), 2):
        shared[(a, b)].add(k)


In [ ]:
P = nx.Graph()
P.add_nodes_from(ids)
for (a, b), ks in shared.items():
    P.add_edge(a, b, weight=len(ks), idf_weight=sum(idf[k] for k in ks), keywords=sorted(ks))
print(f"Граф публикаций: {P.number_of_nodes()} вершин, {P.number_of_edges()} рёбер")


def short(pid, n=70):
    t = title[pid]
    return t if len(t) <= n else t[:n - 1] + "…"


def similar(pid_or_title, k=5, expand=True):
    """Ближайшие к статье публикации на графе. Сначала по весу прямых рёбер
    (число общих ключевых слов); если соседей меньше k — дополняем
    персонализированным PageRank (пути длины 2+ через общих соседей)."""
    pid = pid_or_title if pid_or_title in P else next(
        (i for i in ids if title[i].lower().startswith(pid_or_title.lower())), None)
    if pid is None:
        raise KeyError(f"статья не найдена: {pid_or_title}")
    direct = sorted(P[pid].items(), key=lambda x: (-x[1]["weight"], -x[1]["idf_weight"]))
    out = [(n, d["weight"], d["keywords"], "прямая связь") for n, d in direct[:k]]
    if expand and len(out) < k:
        ppr = nx.pagerank(P, alpha=0.85, personalization={pid: 1.0}, weight="weight")
        have = {n for n, *_ in out} | {pid}
        for n, s in sorted(ppr.items(), key=lambda x: -x[1]):
            if n in have or s <= 1.0 / N * 1.0001:
                continue
            out.append((n, 0, [], f"через соседей (PPR={s:.4f})"))
            have.add(n)
            if len(out) >= k:
                break
    return pid, out


In [ ]:
kw_stems = {k: set(stem(w) for w in k.split()) for k in inv}



_sdf = Counter(s for t in papers.text for s in {stem(w) for w in re.findall(r"[a-z][a-z0-9]+", t.lower())})
stem_idf = lambda s: math.log(N / (1 + _sdf.get(s, 0)))


def query_stems(q: str) -> set:
    s = {s for run in content_runs(q) for s, _ in run}
    return s or {stem(w) for w in re.findall(r"[a-z]+", q.lower())}


def search(q: str, k=10, use_ppr=True, alpha=0.3, ppr_weight="idf_weight"):
    """1) запрос нормализуется так же, как статьи; 2) каждое ключевое слово графа
    получает релевантность = доля его стемов, совпавших с запросом; 3) статья
    получает сумму rel*idf по своим ключевым словам; 4) оценки распространяются
    по графу публикаций персонализированным PageRank, чтобы подтянуть статьи,
    связанные с попавшими в цель, но не содержащие слов запроса."""
    qs = query_stems(q)
    rel = {}
    for w, ks in kw_stems.items():
        hit = qs & ks
        if hit:
            den = sum(max(stem_idf(x), 0.05) for x in ks)
            r = sum(max(stem_idf(x), 0.05) for x in hit) / den
            if r > 0.15:
                rel[w] = r
    direct = defaultdict(float)
    for w, r in rel.items():
        for pid in inv[w]:
            direct[pid] += r * idf[w]
    if not direct:
        return [], {}, rel
    scores = dict(direct)
    if use_ppr:
        tot = sum(direct.values())
        pers = {n: v / tot for n, v in direct.items()}
        ppr = nx.pagerank(P, alpha=0.85, personalization=pers, weight=ppr_weight)
        mx_d, mx_p = max(direct.values()), max(ppr.values())
        scores = {n: (1 - alpha) * direct.get(n, 0) / mx_d + alpha * ppr[n] / mx_p for n in P}
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return ranked, direct, rel


In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = tfidf.fit_transform(papers.text)


def baseline(q, k=10):
    sims = linear_kernel(tfidf.transform([q]), X).ravel()
    return [ids[i] for i in np.argsort(-sims)[:k]]



TESTS = {
    "knowledge graph": r"knowledge graph|\bkgs?\b|graphrag",
    "multimodal image video": r"multimodal|multi-modal|image|visual|video",
    "privacy attacks poisoning": r"privacy|poison|attack|adversarial|security",
    "medical clinical": r"medical|clinical|health|biomedical|patient",
    "recommendation system": r"recommend",
    "agentic agents": r"agent",
    "hallucination faithfulness": r"hallucinat|faithful",
    "multi-hop question answering": r"multi-?hop",
    "code generation software": r"\bcode\b|software|programming",
    "long context window": r"long[- ]context|context window|context length",
}
text_of = dict(zip(papers.id, papers.text.str.lower()))
results = {"queries": {}}
rows = []
for q, pat in TESTS.items():
    rel_set = {i for i in ids if re.search(pat, text_of[i])}
    g, *_ = search(q, 10, use_ppr=True)
    g_no, *_ = search(q, 10, use_ppr=False)
    g10 = [i for i, _ in g]
    g_no10 = [i for i, _ in g_no]
    b10 = baseline(q, 10)
    prec = lambda lst: sum(i in rel_set for i in lst) / max(len(lst), 1)
    rows.append((q, len(rel_set), prec(g_no10), prec(g10), prec(b10)))
    results["queries"][q] = {"relevant_in_corpus": len(rel_set), "P@10_keywords_only": prec(g_no10),
                             "P@10_graph_ppr": prec(g10), "P@10_tfidf_baseline": prec(b10)}


In [ ]:
def mean_p10(**kw_):
    return float(np.mean([
        sum(i in {x for x in ids if re.search(pat, text_of[x])} for i, _ in search(q, 10, **kw_)[0]) / 10
        for q, pat in TESTS.items()]))


ablation = {"только ключевые слова": mean_p10(use_ppr=False)}
for wt in ("weight", "idf_weight"):
    for a in (0.2, 0.3, 0.5, 0.7):
        ablation[f"PPR, вес={wt}, alpha={a}"] = mean_p10(use_ppr=True, alpha=a, ppr_weight=wt)
results["ablation_mean_P@10"] = ablation
mean = lambda j: float(np.mean([r[j] for r in rows]))
results["mean_P@10"] = {"keywords_only": mean(2), "graph_ppr": mean(3), "tfidf_baseline": mean(4)}

rng = np.random.default_rng(0)
cat_cnt = Counter(category.values())
same, tot, rand_p = 0, 0, []
for pid in ids:
    if P.degree(pid) == 0:
        continue
    _, out = similar(pid, 5, expand=False)
    same += sum(category[n] == category[pid] for n, *_ in out)
    tot += len(out)
    rand_p.append((cat_cnt[category[pid]] - 1) / (N - 1))
results["similar_same_category_P@5"] = same / tot
results["similar_random_baseline"] = float(np.mean(rand_p))
sim_full = linear_kernel(X)
idx = {p: i for i, p in enumerate(ids)}
g_cos, t_cos, r_cos = [], [], []
for pid in ids[::4]:
    i = idx[pid]
    _, out = similar(pid, 5, expand=False)
    g_cos.append(np.mean([sim_full[i, idx[n]] for n, *_ in out]))
    row = sim_full[i].copy(); row[i] = -1
    t_cos.append(np.sort(row)[-5:].mean())
    r_cos.append(np.mean(rng.choice(np.delete(sim_full[i], i), 5)))


In [ ]:
results["similar_cosine"] = {"graph_top5": float(np.mean(g_cos)), "tfidf_top5_upper_bound": float(np.mean(t_cos)),
                             "random": float(np.mean(r_cos))}
results["papers_with_neighbors"] = sum(1 for n in ids if P.degree(n) > 0)
results["graph"] = {"nodes": P.number_of_nodes(), "edges": P.number_of_edges(),
                    "isolated": sum(1 for n in ids if P.degree(n) == 0),
                    "avg_degree": 2 * P.number_of_edges() / N,
                    "max_weight": max(d["weight"] for *_, d in P.edges(data=True)),
                    "components": nx.number_connected_components(P)}
(RES / "search_eval.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))

demo = []
demo.append("=== ПОИСК ПОХОЖИХ ПУБЛИКАЦИЙ ===")
heavy = sorted(P.edges(data=True), key=lambda e: -e[2]["weight"])
for pid in [heavy[0][0], papers.id[3], papers.id[100]]:
    src, out = similar(pid, 5)
    demo.append(f"\nИСХОДНАЯ: {title[src]}\n  ключевые слова: {kw[src]}")
    for n, w, ks, how in out:
        demo.append(f"  [{w}] {title[n]}\n        общие: {ks if ks else '—'} ({how})")
demo.append("\n=== ПОИСК ПО ЗАПРОСУ ===")
for q in ["knowledge graph question answering", "multimodal retrieval images", "poisoning attacks defense",
          "hallucination detection", "medical clinical decision support", "recommendation personalized users", "рекомендательные системы"]:
    ranked, direct, rel = search(q, 5)
    if not ranked:
        demo.append(f"\nЗАПРОС: «{q}» — нет совпадений (словарь корпуса англоязычный)")
        continue
    demo.append(f"\nЗАПРОС: «{q}»  (совпавших ключевых слов графа: {len(rel)})")
    for pid, s in ranked:
        tag = "прямое" if pid in direct else "через граф"
        demo.append(f"  {s:.3f} [{tag}] {title[pid]}")
(RES / "search_demo.txt").write_text("\n".join(demo))
print("\n".join(demo))
print(json.dumps(results, ensure_ascii=False, indent=1))


fig, ax = plt.subplots(1, 2, figsize=(16, 6))


degs = [d for _, d in P.degree()]
sns.histplot(degs, bins=40, ax=ax[0], color="#4C72B0"); ax[0].set_yscale("log")
ax[0].set(title="Степень вершин в графе публикаций", xlabel="Соседей", ylabel="Статей (log)")
w = Counter(d["weight"] for *_, d in P.edges(data=True))
ax[1].bar(list(w.keys()), list(w.values()), color="#DD8452"); ax[1].set_yscale("log")
ax[1].set(title="Вес ребра = число общих ключевых слов", xlabel="Общих слов", ylabel="Рёбер (log)")
plt.tight_layout(); plt.savefig(FIG / "11_paper_graph_stats.png", dpi=150); plt.show(); plt.close()


In [ ]:
q = "knowledge graph question answering"
ranked, direct, _ = search(q, 12)
top = [p for p, _ in ranked]
S = P.subgraph(top).copy()
fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(S, seed=3, k=1.2, weight="weight")
nx.draw_networkx_edges(S, pos, width=[1 + 1.5 * d["weight"] for *_, d in S.edges(data=True)], alpha=.5, ax=ax)
nx.draw_networkx_nodes(S, pos, node_size=900, ax=ax,
                       node_color=["#55A868" if n in direct else "#C44E52" for n in S], edgecolors="k")
nx.draw_networkx_labels(S, pos, {n: f"{i+1}" for i, n in enumerate(top) if n in S}, font_color="w", ax=ax)
ax.set_title(f"Топ-12 по запросу «{q}» и связи между ними\n(зелёный — прямое совпадение, красный — найдено через граф)")
ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "12_search_subgraph.png", dpi=150); plt.show(); plt.close()

fig, ax = plt.subplots(figsize=(13, 6.5))
x = np.arange(len(rows)); wd = 0.27
ax.bar(x - wd, [r[2] for r in rows], wd, label="только ключевые слова")
ax.bar(x, [r[3] for r in rows], wd, label="ключевые слова + граф (PPR)")
ax.bar(x + wd, [r[4] for r in rows], wd, label="TF-IDF по тексту (baseline)")
ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=35, ha="right")
ax.set(ylabel="Precision@10", title="Качество поиска по запросу", ylim=(0, 1.05)); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "13_search_eval.png", dpi=150); plt.show(); plt.close()

for *_, d in P.edges(data=True):
    d["keywords"] = "; ".join(d["keywords"])
nx.write_gexf(P, RES / "paper_graph.gexf")
